# Act 6 — The punchline

> **Opening question:** *"We started by asking: what actually drives house prices in Ames beyond just location? We've now looked at distributions, correlations, time trends, and outliers. So — what's the answer?"*

---
This final act assembles everything into one summary dashboard and answers the central question
in plain English. No jargon. Just the story the data told us.

In [ ]:
import sys
sys.path.append('..')
from src.utils import *
from plotly.subplots import make_subplots
import plotly.graph_objects as go

act_header(
    act_num=6,
    title="The punchline",
    opening_question="What actually drives house prices in Ames — and what's the one thing that surprised us most?"
)

df = load_processed('cleaned.csv')

## 6.1 — The five key findings: a summary dashboard

In [ ]:
numeric_df = df.select_dtypes(include='number')
top_corr   = numeric_df.corr()['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False).head(5)

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Top 5 price drivers',
        'Quality → price staircase',
        'Price spread by neighbourhood',
        'Price distribution shape',
        'Year sold vs median price',
        'Size vs price (overall)'
    ),
    vertical_spacing=0.18,
    horizontal_spacing=0.1
)

# 1 — Top correlations
fig.add_trace(go.Bar(
    x=top_corr.values[::-1], y=top_corr.index[::-1],
    orientation='h',
    marker_color='#534AB7',
    text=[f'{v:.2f}' for v in top_corr.values[::-1]],
    textposition='outside'
), row=1, col=1)

# 2 — Quality staircase
qual_med = df.groupby('OverallQual')['SalePrice'].median()
fig.add_trace(go.Bar(
    x=qual_med.index, y=qual_med.values / 1000,
    marker_color='#1D9E75',
    showlegend=False
), row=1, col=2)

# 3 — Neighbourhood prices (top 10)
if 'Neighborhood' in df.columns:
    nbr_med = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).head(10)
    fig.add_trace(go.Bar(
        x=nbr_med.values[::-1] / 1000,
        y=nbr_med.index[::-1],
        orientation='h',
        marker_color='#D85A30',
        showlegend=False
    ), row=1, col=3)

# 4 — Price histogram
import numpy as np
hist_vals, hist_bins = np.histogram(df['SalePrice'].dropna(), bins=40)
fig.add_trace(go.Bar(
    x=hist_bins[:-1] / 1000,
    y=hist_vals,
    marker_color='#AFA9EC',
    showlegend=False
), row=2, col=1)

# 5 — Year trend
if 'YrSold' in df.columns:
    yr_med = df.groupby('YrSold')['SalePrice'].median()
    fig.add_trace(go.Scatter(
        x=yr_med.index, y=yr_med.values / 1000,
        mode='lines+markers',
        line=dict(color='#BA7517', width=2.5),
        marker=dict(size=8),
        showlegend=False
    ), row=2, col=2)

# 6 — Size vs price scatter (sample)
sample = df.sample(min(500, len(df)), random_state=42)
fig.add_trace(go.Scatter(
    x=sample['GrLivArea'], y=sample['SalePrice'] / 1000,
    mode='markers',
    marker=dict(color='#534AB7', size=4, opacity=0.5),
    showlegend=False
), row=2, col=3)

fig.update_layout(
    title=dict(text='Ames Housing EDA — six-act story summary dashboard', font_size=15),
    template='plotly_white',
    height=650,
    margin=dict(t=80, b=40)
)

save_plotly(fig, 'act6_summary_dashboard.html')
fig.show()

## 6.2 — KPI scorecard: the numbers that matter

In [ ]:
median_price  = df['SalePrice'].median()
max_price     = df['SalePrice'].max()
min_price     = df['SalePrice'].min()
top_driver    = numeric_df.corr()['SalePrice'].drop('SalePrice').abs().idxmax()
top_r         = numeric_df.corr()['SalePrice'].drop('SalePrice').abs().max()
n_outliers    = int(((df['SalePrice'] - df['SalePrice'].median()).abs() > 1.5 * (df['SalePrice'].quantile(0.75) - df['SalePrice'].quantile(0.25))).sum())

print("\n" + "═" * 55)
print("  AMES HOUSING EDA — FINAL SCORECARD")
print("═" * 55)
print(f"  Total properties analysed : {len(df):,}")
print(f"  Median sale price         : ${median_price:,.0f}")
print(f"  Price range               : ${min_price:,.0f} — ${max_price:,.0f}")
print(f"  #1 price driver           : {top_driver} (r = {top_r:.2f})")
print(f"  Price outliers detected   : {n_outliers}")
if 'Neighborhood' in df.columns:
    top_nbr = df.groupby('Neighborhood')['SalePrice'].median().idxmax()
    print(f"  Most expensive area       : {top_nbr}")
print("═" * 55)

## 6.3 — The answer, in plain English

In [ ]:
insight_callout(
    "FINDING 1 — Quality beats size.\n"
    "  OverallQual is the strongest predictor of SalePrice (r ≈ 0.79).\n"
    "  Two houses of identical size can differ by $100k+ based on quality rating alone.\n\n"

    "FINDING 2 — Size is the second act.\n"
    "  GrLivArea and TotalBsmtSF together explain most of the remaining price variation.\n"
    "  Every extra 500 sq ft adds roughly $25–40k to the expected price.\n\n"

    "FINDING 3 — Neighbourhood is a silent premium.\n"
    "  A house in the top neighbourhood vs the bottom can differ by $150k+\n"
    "  at the same quality and size. Location is not just geography — it's identity.\n\n"

    "FINDING 4 — Ames survived the 2008 crash.\n"
    "  Sales volume fell sharply but median prices held. A university town\n"
    "  acts as an economic anchor that purely residential markets don't have.\n\n"

    "FINDING 5 — Two outliers are almost certainly data errors.\n"
    "  Two large-area, low-price entries defy every pattern in the dataset.\n"
    "  They should be removed before any price prediction model is built.",
    label="The five things this data told us"
)

## 6.4 — The one sentence that answers everything

In [ ]:
punchline(
    "In Ames, Iowa — quality of construction is the single biggest driver of house price, "
    "size amplifies it, neighbourhood multiplies both, "
    "and even a market crash couldn't break the pattern."
)

---
## What comes next?

This EDA is the foundation. The natural next step is to build a **regression model** that uses `OverallQual`, `GrLivArea`, `TotalBsmtSF`, `Neighborhood`, and `YearBuilt` to **predict SalePrice** — which is exactly what the Kaggle competition asks for.

The insights from this EDA already tell you:
- Which features to prioritise
- Which outliers to remove before modelling
- That log-transforming `SalePrice` will give a better model fit

**Every good model starts with a great EDA. You've just done the great EDA.**